In [ ]:
import time
from pathlib import Path
from pyspark.sql import functions as F

def get_storage_info(delta_path: Path):
    """Calculates disk footprint of a Delta table."""
    if not delta_path.exists():
        return "N/A"
    parquet_files = list(delta_path.glob("**/*.parquet"))
    total_bytes = sum(f.stat().st_size for f in parquet_files)
    return f"{total_bytes / 1024:.1f} KB ({len(parquet_files)} files)"

def evaluate_query(query_name, df_baseline=None, df_optimized=None, data_product_path=None, aqe_query=None):
    print(f"\n{'='*60}")
    print(f"EVALUATING: {query_name}")
    print(f"{'='*60}")
    
    if aqe_query is not None:
        spark.conf.set("spark.sql.adaptive.enabled", "false")
        df_baseline = spark.sql(aqe_query)
        print("\n--- Baseline Physical Plan (AQE Disabled) ---")
        df_baseline.explain("formatted")
        t0 = time.time()
        base_rows = df_baseline.count()
        base_time = time.time() - t0
        
        spark.conf.set("spark.sql.adaptive.enabled", "true")
        spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
        df_optimized = spark.sql(aqe_query)
        t0 = time.time()
        opt_rows = df_optimized.count()
        opt_time = time.time() - t0
        print("\n--- Optimized Physical Plan (AQE Enabled) ---")
        df_optimized.explain("formatted")
        storage_str = "N/A (AQE Engine Optimization on Integrated Table)"
    else:
        print("\n--- Baseline Physical Plan (On-Demand) ---")
        df_baseline.explain("formatted")
        print("\n--- Optimized Physical Plan (Data Product) ---")
        df_optimized.explain("formatted")
        
        t0 = time.time()
        base_rows = df_baseline.count()
        base_time = time.time() - t0
        
        t0 = time.time()
        opt_rows = df_optimized.count()
        opt_time = time.time() - t0
        storage_str = get_storage_info(data_product_path) if data_product_path else "N/A"

    assert base_rows == opt_rows, f"Row count mismatch: {base_rows} vs {opt_rows}"
    cols = df_baseline.columns
    assert df_baseline.orderBy(*cols).collect() == df_optimized.orderBy(*cols).collect(), "Result data mismatch!"
    print("\n[VERIFIED] Optimized query produces identical results to baseline.")
    
    speedup = ((base_time - opt_time) / base_time) * 100 if base_time > 0 else 0
    print(f"\n--- Performance Summary ---")
    print(f"Result Rows:       {base_rows:,}")
    print(f"Baseline Time:     {base_time:.2f}s")
    print(f"Optimized Time:    {opt_time:.2f}s")
    print(f"Speedup:           {speedup:.1f}%")
    print(f"Storage Overhead:  {storage_str}\n")

In [8]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark
from src.lake import GOLD, SILVER, read_delta

spark = create_spark("benchmark")
read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView("integrated_taxi_trips")

26/09/20 14:18:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [9]:
query_1 = """
WITH monthly_zone_trips AS (
    SELECT
        TRUNC(pickup_date, 'MM') AS trip_month,
        pickup_location_id,
        pickup_borough,
        pickup_zone,
        pickup_date
    FROM integrated_taxi_trips
    WHERE pickup_zone != 'UNKNOWN'
      AND pickup_date IS NOT NULL
)
SELECT
    trip_month,
    pickup_location_id,
    pickup_borough,
    pickup_zone,
    COUNT(*) AS total_trips,
    COUNT(DISTINCT pickup_date) AS active_days,
    ROUND(
        CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT pickup_date),
        2
    ) AS avg_daily_trips
FROM monthly_zone_trips
GROUP BY trip_month, pickup_location_id, pickup_borough, pickup_zone
ORDER BY trip_month ASC, total_trips DESC
"""
query_2 = """
WITH binned_weather AS (
    SELECT
        trip_distance,
        CASE
            WHEN temperature_c IS NULL THEN 'Unknown'
            WHEN temperature_c < 0 THEN 'Freezing (<0°C)'
            WHEN temperature_c BETWEEN 0 AND 10 THEN 'Cold (0°C to 10°C)'
            WHEN temperature_c BETWEEN 10.01 AND 20 THEN 'Moderate (10°C to 20°C)'
            ELSE 'Warm (>20°C)'
        END AS temp_category,
        CASE
            WHEN wind_speed_ms IS NULL THEN 'Unknown'
            WHEN wind_speed_ms < 2 THEN 'Calm (<2 m/s)'
            WHEN wind_speed_ms BETWEEN 2 AND 6 THEN 'Moderate Wind (2-6 m/s)'
            ELSE 'High Wind (>6 m/s)'
        END AS wind_category
    FROM integrated_taxi_trips
    WHERE trip_distance > 0 AND trip_distance < 100
)
SELECT
    temp_category,
    wind_category,
    COUNT(*) AS trip_count,
    ROUND(AVG(trip_distance), 2) AS avg_distance_miles
FROM binned_weather
GROUP BY temp_category, wind_category
ORDER BY temp_category, wind_category
"""
query_3 = """
WITH rounded_pm25 AS (
    SELECT
        ROUND(pm25, 0) AS pm25_level,
        pickup_date,
        pickup_hour
    FROM integrated_taxi_trips
    WHERE pm25 IS NOT NULL AND pickup_date IS NOT NULL AND pickup_hour IS NOT NULL
)
SELECT
    pm25_level,
    COUNT(*) AS trips,
    COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS observed_hours,
    ROUND(CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT struct(pickup_date, pickup_hour)), 2) AS trips_per_hour
FROM rounded_pm25
GROUP BY pm25_level
ORDER BY pm25_level DESC
"""
query_4 = """
WITH trips_with_weather AS (
    SELECT 
        pickup_zone,
        pickup_date,
        pickup_hour,
        NTILE(4) OVER (
            PARTITION BY pickup_zone
            ORDER BY (10 * sqrt(wind_speed_ms) - wind_speed_ms + 10.5) * (33 - temperature_c)
        ) AS weather_condition
    FROM integrated_taxi_trips
    WHERE pickup_zone IS NOT NULL AND pickup_date IS NOT NULL AND pickup_hour IS NOT NULL
),
hourly_demand AS (
    SELECT 
        pickup_zone,
        weather_condition,
        COUNT(1) / COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS trips_per_hour
    FROM trips_with_weather
    GROUP BY pickup_zone, weather_condition
),
pivoted AS (
    SELECT * FROM hourly_demand
    PIVOT (
        ROUND(AVG(trips_per_hour), 2)
        FOR weather_condition IN (1 AS coldest, 2 AS cool, 3 AS warm, 4 AS warmest)
    )
)
SELECT 
    pickup_zone,
    coldest, cool, warm, warmest,
    ROUND(((GREATEST(coldest, cool, warm, warmest) - LEAST(coldest, cool, warm, warmest)) / ((coldest + cool + warm + warmest) / 4.0)) * 100, 2) AS pct_variation
FROM pivoted
WHERE (coldest + cool + warm + warmest) / 4.0 >= 10
ORDER BY pct_variation DESC
"""
query_5 = """
WITH hourly_by_dow AS (
    SELECT
        CASE dayofweek(pickup_date)
            WHEN 1 THEN 'Sunday'
            WHEN 2 THEN 'Monday'
            WHEN 3 THEN 'Tuesday'
            WHEN 4 THEN 'Wednesday'
            WHEN 5 THEN 'Thursday'
            WHEN 6 THEN 'Friday'
            WHEN 7 THEN 'Saturday'
        END AS day_of_week,
        CASE dayofweek(pickup_date)
            WHEN 1 THEN 7
            ELSE dayofweek(pickup_date) - 1
        END AS dow_order,
        pickup_hour,
        COUNT(*) AS trip_count,
        COUNT(DISTINCT pickup_date) AS active_days,
        ROUND(
            CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT pickup_date),
            2
        ) AS avg_trips
    FROM integrated_taxi_trips
    WHERE pickup_date IS NOT NULL
      AND pickup_hour IS NOT NULL
    GROUP BY dayofweek(pickup_date), pickup_hour
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY day_of_week
            ORDER BY avg_trips DESC, trip_count DESC
        ) AS peak_rank
    FROM hourly_by_dow
)
SELECT
    day_of_week,
    pickup_hour AS peak_hour,
    trip_count,
    active_days,
    avg_trips
FROM ranked
WHERE peak_rank = 1
ORDER BY dow_order
"""
query_6 = """
WITH monthly AS (
    SELECT
        TRUNC(pickup_date, 'MM') AS trip_month,
        COUNT(*) AS total_trips,
        COUNT(DISTINCT pickup_date) AS active_days,
        ROUND(
            CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT pickup_date),
            2
        ) AS avg_daily_trips
    FROM integrated_taxi_trips
    WHERE pickup_date IS NOT NULL
    GROUP BY TRUNC(pickup_date, 'MM')
)
SELECT
    trip_month,
    total_trips,
    active_days,
    avg_daily_trips,
    LAG(avg_daily_trips) OVER (ORDER BY trip_month) AS prev_month_avg_daily,
    ROUND(
        100.0 * (
            avg_daily_trips - LAG(avg_daily_trips) OVER (ORDER BY trip_month)
        ) / LAG(avg_daily_trips) OVER (ORDER BY trip_month),
        2
    ) AS pct_change_vs_prev_month
FROM monthly
ORDER BY trip_month
"""


### Benchmark Query 1

In [10]:
# Baseline: Full scan over 9.4M rows on demand
df_q1_base = spark.sql(query_1)

# Optimized: Read from pre-materialized Data Product
prod_path = GOLD / "data_products" / "taxi_zone_monthly_demand"
df_q1_opt = read_delta(spark, prod_path).select(
    "trip_month", "pickup_location_id", "pickup_borough", "pickup_zone",
    "total_trips", "active_days", "avg_daily_trips"
)

evaluate_query("Query 1: Monthly Taxi Demand per Zone", df_q1_base, df_q1_opt, prod_path)


EVALUATING: Query 1: Monthly Taxi Demand per Zone

--- Baseline Physical Plan (On-Demand) ---
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Exchange (8)
            +- HashAggregate (7)
               +- HashAggregate (6)
                  +- Exchange (5)
                     +- HashAggregate (4)
                        +- Project (3)
                           +- Filter (2)
                              +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [pickup_location_id#1836, pickup_zone#1837, pickup_borough#1838, pickup_date#1850]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/data/lake/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_date#1850)]
PushedFilters: [IsNotNull(pickup_zone), Not(EqualTo(pickup_zone,UNKNOWN))]
ReadSchema: struct<pickup_location_id:int,pickup_zone:string,pickup_borough:string>

(2) Filter
Input [4


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       775
Baseline Time:     1.37s
Optimized Time:    0.23s
Speedup:           83.4%
Storage Overhead:  69.0 KB (8 files)



### Benchmark Query 2

In [11]:
# Baseline: On-demand CASE binning over 9.4M rows
df_q2_base = spark.sql(query_2)
# Optimized: Read from Data Product
prod_path = GOLD / "data_products" / "weather_impact_summary"
df_q2_opt = read_delta(spark, prod_path).select(
    "temp_category", "wind_category", "trip_count", "avg_distance_miles"
)
evaluate_query("Query 2: Average Distance by Weather", df_q2_base, df_q2_opt, prod_path)


EVALUATING: Query 2: Average Distance by Weather

--- Baseline Physical Plan (On-Demand) ---
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [trip_distance#1835, temperature_c#1846, wind_speed_ms#1847, pickup_date#1850]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/data/lake/gold/integrated_taxi_trips]
PushedFilters: [IsNotNull(trip_distance), GreaterThan(trip_distance,0.0), LessThan(trip_distance,100.0)]
ReadSchema: struct<trip_distance:double,temperature_c:double,wind_speed_ms:double>

(2) Filter
Input [4]: [trip_distance#1835, temperature_c#1846, wind_speed_ms#1847, pickup_date#1850]
Condition : ((isnotnull(trip_distance#1835) AND (trip_distance#1835 > 0.0)) AND (tri


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       12
Baseline Time:     1.25s
Optimized Time:    0.24s
Speedup:           81.1%
Storage Overhead:  6.1 KB (2 files)



### Benchmark Query 3

In [12]:
### Benchmark Query 3: Air Quality vs Demand
df_q3_base = spark.sql(query_3)
prod_path_q3 = GOLD / "data_products" / "air_quality_demand_summary"
df_q3_opt = read_delta(spark, prod_path_q3).select(
    "pm25_level", "trips", "observed_hours", "trips_per_hour"
)

evaluate_query("Query 3: Air Quality vs Demand", df_q3_base, df_q3_opt, prod_path_q3)


EVALUATING: Query 3: Air Quality vs Demand

--- Baseline Physical Plan (On-Demand) ---
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Exchange (8)
            +- HashAggregate (7)
               +- HashAggregate (6)
                  +- Exchange (5)
                     +- HashAggregate (4)
                        +- Project (3)
                           +- Filter (2)
                              +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pm25#1848, pickup_hour#1851, pickup_date#1850]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/data/lake/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_date#1850)]
PushedFilters: [IsNotNull(pm25), IsNotNull(pickup_hour)]
ReadSchema: struct<pm25:double,pickup_hour:int>

(2) Filter
Input [3]: [pm25#1848, pickup_hour#1851, pickup_date#1850]
Condition : (isnotnull(pm25#1848) AND isnotnu


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       33
Baseline Time:     1.72s
Optimized Time:    0.19s
Speedup:           89.0%
Storage Overhead:  3.3 KB (1 files)



### Benchmark Query 4

In [13]:
# Baseline: On-demand NTILE + distinct struct + PIVOT
df_q4_base = spark.sql(query_4)

# Optimized: Read pre-computed sensitivity table
prod_path = GOLD / "data_products" / "zone_weather_sensitivity"
df_q4_opt = read_delta(spark, prod_path).select(
    "pickup_zone", "coldest", "cool", "warm", "warmest", "pct_variation"
)

evaluate_query("Query 4: Zone Weather Demand Variation", df_q4_base, df_q4_opt, prod_path)


EVALUATING: Query 4: Zone Weather Demand Variation

--- Baseline Physical Plan (On-Demand) ---
== Physical Plan ==
AdaptiveSparkPlan (20)
+- Sort (19)
   +- Exchange (18)
      +- Project (17)
         +- Filter (16)
            +- HashAggregate (15)
               +- HashAggregate (14)
                  +- HashAggregate (13)
                     +- HashAggregate (12)
                        +- HashAggregate (11)
                           +- HashAggregate (10)
                              +- HashAggregate (9)
                                 +- HashAggregate (8)
                                    +- Project (7)
                                       +- Window (6)
                                          +- Sort (5)
                                             +- Exchange (4)
                                                +- Project (3)
                                                   +- Filter (2)
                                                      +- Scan parquet  (1)


(1) 


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       53
Baseline Time:     5.41s
Optimized Time:    0.34s
Speedup:           93.7%
Storage Overhead:  10.7 KB (2 files)



### Benchmark Query 5

In [ ]:
evaluate_query("Query 5: Peak Travel Hours by Day of Week", aqe_query=query_5)


EVALUATING: Query 5: Peak Travel Hours by Day of Week

--- Baseline Physical Plan (AQE Disabled) ---
== Physical Plan ==
* Project (21)
+- * Sort (20)
   +- Exchange (19)
      +- * Project (18)
         +- * Filter (17)
            +- Window (16)
               +- WindowGroupLimit (15)
                  +- * Sort (14)
                     +- Exchange (13)
                        +- WindowGroupLimit (12)
                           +- * Sort (11)
                              +- * HashAggregate (10)
                                 +- Exchange (9)
                                    +- * HashAggregate (8)
                                       +- * HashAggregate (7)
                                          +- Exchange (6)
                                             +- * HashAggregate (5)
                                                +- * Project (4)
                                                   +- * Filter (3)
                                                      +- * Columnar

### Benchmark Query 6

In [ ]:
evaluate_query("Query 6: Monthly Trends in Taxi Demand", aqe_query=query_6)


EVALUATING: Query 6: Monthly Trends in Taxi Demand

--- Baseline Physical Plan (AQE Disabled) ---


26/09/20 14:20:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


== Physical Plan ==
* Project (13)
+- Window (12)
   +- * Sort (11)
      +- Exchange (10)
         +- * HashAggregate (9)
            +- Exchange (8)
               +- * HashAggregate (7)
                  +- * HashAggregate (6)
                     +- Exchange (5)
                        +- * HashAggregate (4)
                           +- * Project (3)
                              +- * ColumnarToRow (2)
                                 +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [pickup_date#1850]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/data/lake/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_date#1850)]
ReadSchema: struct<>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [pickup_date#1850]

(3) Project [codegen id : 1]
Output [2]: [pickup_date#1850, trunc(pickup_date#1850, MM) AS _groupingexpression#9727]
Input [1]: [pickup_date#1850]

(4) HashAggregate [codegen id : 1]
Input 

26/09/20 14:20:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/20 14:20:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/20 14:20:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/20 14:20:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/20 14:20:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/20 14:20:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/20 1


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       4
Baseline Time:     0.52s
Optimized Time:    0.17s
Speedup:           66.6%
Storage Overhead:  N/A (AQE Engine Optimization on Integrated Table)



26/09/20 14:20:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/20 14:20:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/20 14:20:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/20 14:20:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/20 14:20:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/20 14:20:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
